In [1]:
import os
import re
import glob
import pickle 
import numpy as np
import netCDF4 as nc
import h5py

import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight

In [2]:
base_path_l1 = "/Volumes/EXTERNO-MP/TEMPO_l1"
l1_files = glob.glob(f"{base_path_l1}/*.nc")
base_path_l2 = "../data/raw_l2_v3"
out_path = "../data/TEMPO_cs_RGB_all"
band = "band_290_490_nm"
version = "V03"

cloud_threshold = 0.3
shadow_threshold = 0.05

In [3]:
banned_objs = [
"TEMPO_RAD_L1_V04_20250909T190538Z_S010G01-001",
"TEMPO_RAD_L1_V04_20250909T221218Z_S013G02-018",
"TEMPO_RAD_L1_V04_20250909T234546Z_S015G01-030",
"TEMPO_RAD_L1_V04_20250909T121157Z_S002G05-092",
"TEMPO_RAD_L1_V04_20250909T220538Z_S013G01-015",
"TEMPO_RAD_L1_V04_20250909T195538Z_S011G01-058",
"TEMPO_RAD_L1_V04_20250909T191846Z_S010G03-005",
"TEMPO_RAD_L1_V04_20250909T200846Z_S011G03-054",
"TEMPO_RAD_L1_V04_20250909T191212Z_S010G02-006",
"TEMPO_RAD_L1_V04_20250909T210538Z_S012G01-008",
"TEMPO_RAD_L1_V04_20250909T125842Z_S003G06-094",
"TEMPO_RAD_L1_V04_20250909T235906Z_S015G03-003",
"TEMPO_RAD_L1_V04_20250909T230538Z_S014G01-028",
"TEMPO_RAD_L1_V04_20250909T135205Z_S004G08-100",
"TEMPO_RAD_L1_V04_20250909T200212Z_S011G02-056",
"TEMPO_RAD_L1_V04_20250909T135842Z_S004G09-099",
"TEMPO_RAD_L1_V04_20250909T121834Z_S002G06-091",
"TEMPO_RAD_L1_V04_20250909T235226Z_S015G02-004",
"TEMPO_RAD_L1_V04_20250909T134528Z_S004G07-060",
]
print(len(l1_files))
l1_files = [file for file in l1_files if file.split("/")[-1].split(".")[0] not in banned_objs]
print(len(l1_files))

98
79


In [4]:
def extract_timestamp_scan(filename):
    """Extract timestamp and scan/granule info from filename"""
    # Pattern: TEMPO_{PRODUCT}_L{LEVEL}_V04_{TIMESTAMP}_S{SCAN}G{GRANULE}
    match = re.search(r'(\d{8}T\d{6}Z)_S(\d{3})G(\d{2})', filename)
    if match:
        return match.group(1), match.group(2), match.group(3)
    return None, None, None

def find_matching_l2_file(l1_file, l2_dir='L2', version="V03"):
    """Find corresponding L2 cloud file for an L1 file"""
    timestamp, scan, granule = extract_timestamp_scan(l1_file)
    print(l1_file, timestamp, scan, granule)
    if not timestamp:
        return None
    
    # L2 cloud files have format: TEMPO_CLDO4_L2_V04_{TIMESTAMP}_S{SCAN}G{GRANULE}.nc
    # Note: L2 files might not have the -034 suffix that L1 has
    pattern = f"TEMPO_CLDO4_L2_{version}_{timestamp}_S{scan}G{granule.split('-')[0]}.nc"
    l2_path = os.path.join(l2_dir, pattern)
    
    if os.path.exists(l2_path):
        return l2_path
    
    # Try with wildcard if exact match not found
    search_pattern = os.path.join(l2_dir, f"TEMPO_CLDO4_L2_{version}_{timestamp}_S{scan}G*.nc")
    matches = glob.glob(search_pattern)
    return matches[0] if matches else None

In [5]:
def get_l2_parameters(l2_file):
    """
    Compute potential cloud shadow mask using L2 cloud fraction and geolocation data
    
    Parameters:
    -----------
    nc_file : str
        Path to L1B NetCDF file
    l2_file : str  
        Path to L2 file (HDF5 or NetCDF)
    band : str
        Band name for L1B data
    cloud_threshold : float
        Cloud fraction threshold (default 0.05 as in DARCLOS)
    """        
    with nc.Dataset(l2_file) as f:
        solar_zenith = f['geolocation']['solar_zenith_angle'][:]
        solar_zenith = solar_zenith.filled(fill_value=np.nan)
        
        solar_azimuth = f['geolocation']['solar_azimuth_angle'][:]
        solar_azimuth = solar_azimuth.filled(fill_value=np.nan)
        
        viewing_zenith = f['geolocation']['viewing_zenith_angle'][:]
        viewing_zenith = viewing_zenith.filled(fill_value=np.nan)
        
        viewing_azimuth = f['geolocation']['viewing_azimuth_angle'][:]
        viewing_azimuth = viewing_azimuth.filled(fill_value=np.nan)
        
        relative_azimuth = f['geolocation']['relative_azimuth_angle'][:]
        relative_azimuth = relative_azimuth.filled(fill_value=np.nan)

        GLER466 = f['support_data']['GLER466'][:]
        GLER466 = GLER466.filled(fill_value=0)
                
    # Use L2 angles (more accurate)
    solar_zenith = (solar_zenith + 180) / 360    
    viewing_zenith = (viewing_zenith + 180) / 360
    relative_azimuth = (relative_azimuth + 180) / 360

    solar_zenith = np.nan_to_num(solar_zenith, nan=0.0)
    viewing_zenith = np.nan_to_num(viewing_zenith, nan=0.0)
    relative_azimuth = np.nan_to_num(relative_azimuth, nan=0.0)
    return solar_zenith, viewing_zenith, relative_azimuth, GLER466

In [6]:
file_list = []
labels = []
plot = False

for l1_file in l1_files:
    # Find matching L2 file
    l2_file = find_matching_l2_file(l1_file, base_path_l2, version)

    sza, vza, raa, GLER466 = get_l2_parameters(l2_file)

    sza = np.flip(np.transpose(sza, (1, 0)), axis=0)
    vza = np.flip(np.transpose(vza, (1, 0)), axis=0)
    raa = np.flip(np.transpose(raa, (1, 0)), axis=0)
    GLER466 = np.flip(np.transpose(GLER466, (1, 0)), axis=0)

    if not l2_file:
        print(f"  WARNING: No matching L2 file found, skipping...")
        continue

    # Read RGB data from L1
    with h5py.File(l1_file, 'r') as f:
        red = f['cloud_mask_group']['red'][:]
        green = f['cloud_mask_group']['green'][:]
        blue = f['cloud_mask_group']['blue'][:]
        
    # Clip RGB values exceeding 1.0
    red = np.where((red > 1.0), 1.0, red)[:,:,np.newaxis]
    green = np.where((green > 1.0), 1.0, green)[:,:,np.newaxis]
    blue = np.where((blue > 1.0), 1.0, blue)[:,:,np.newaxis]
    
    # Transpose and flip for correct orientation
    red = np.flip(np.transpose(red, (1, 0, 2)), axis=0)
    green = np.flip(np.transpose(green, (1, 0, 2)), axis=0)
    blue = np.flip(np.transpose(blue, (1, 0, 2)), axis=0)

    # Convert to 8-bit (0-255)
    red_8bit = (red * 255).astype(np.uint8)
    green_8bit = (green * 255).astype(np.uint8)
    blue_8bit = (blue * 255).astype(np.uint8)
    
    rgb = np.concatenate((red_8bit, green_8bit, blue_8bit), axis=2)


    sza_ = sza[:,:,np.newaxis]
    vza_ = vza[:,:,np.newaxis]
    raa_ = raa[:,:,np.newaxis]
    GLER466 = GLER466[:,:,np.newaxis]

    all_comb = np.concatenate((sza_, vza_, raa_, GLER466), axis=2)    

    out_file = l1_file.split("/")[-1].replace('.nc', '.npy')
    if plot:

        print(sza.min(), sza.max(), sza.mean())
        print(vza.min(), vza.max(), vza.mean())
        print(raa.min(), raa.max(), raa.mean())
        print(GLER466.min(), GLER466.max(), GLER466.mean())
        
        rgb_nonan = np.nan_to_num(rgb, nan=0.0)
        # Plot the figure.
        fig, ax = plt.subplots(1,5, figsize=(12, 6))
        ax[0].pcolormesh(rgb_nonan)
        ax[1].pcolormesh(GLER466)
        ax[2].pcolormesh(sza)
        ax[3].pcolormesh(vza)
        ax[4].pcolormesh(raa)
        fig.tight_layout()
    
        for a in ax:
            a.axis('off')
        plt.show()
        plt.close()


        plt.hist(GLER466.flatten(), bins=100)
        plt.show()
    else:
        np.save(f"{out_path}/complementary/{out_file}", combined)
print("all good")

/Volumes/EXTERNO-MP/TEMPO_l1/TEMPO_RAD_L1_V04_20250909T223850Z_S013G06-017.nc 20250909T223850Z 013 06


NameError: name 'combined' is not defined